# Iran Internet Connectivity — Disruption Analysis (Cloudflare Radar)

This notebook uses the **Cloudflare Radar API** to monitor internet traffic and
connectivity in **Iran (`IR`)** and to surface **sudden drops or outages**.

Government-imposed internet shutdowns are a well-documented tactic during periods
of civil unrest: when traffic from an entire country collapses abruptly — outside
of normal daily patterns — it can indicate a deliberate, large-scale disruption.
Organizations such as Cloudflare Radar, NetBlocks, OONI, and the IODA project
track exactly these signals to support digital-rights and transparency reporting.

**What this notebook does**
1. Pulls Iran's HTTP request and network-traffic time series.
2. Plots them so daily rhythms and anomalies are visible.
3. Runs a simple anomaly detector that flags abnormal traffic *drops*.
4. Fetches Cloudflare's curated **outage annotations** for Iran.
5. Pulls the **Internet Quality Index** (latency / bandwidth) as a secondary signal.
6. Summarizes suspected disruption windows.

> **Scope & ethics.** This is an observational, read-only analysis of publicly
> reported aggregate network data. A traffic drop is *evidence* of disruption, not
> proof of intent — confirm against multiple sources (NetBlocks, OONI, news
> reporting) before drawing conclusions.

## 1. Setup

Install dependencies (safe to re-run) and import libraries.

In [ ]:
# Run once; comment out after the first execution.
%pip install -q requests pandas matplotlib

In [ ]:
import os
import datetime as dt
from typing import Optional

import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.figsize'] = (13, 4.5)
plt.rcParams['axes.grid'] = True

## 2. API token

Get a token from the [Cloudflare dashboard](https://dash.cloudflare.com/profile/api-tokens)
— the **Radar** read permission is enough. **Do not paste it into the notebook.**
Set it as an environment variable before launching Jupyter:

```bash
export CLOUDFLARE_API_TOKEN="your_token_here"
```

The cell below reads it from the environment (and falls back to a prompt).

In [ ]:
API_TOKEN = os.environ.get('CLOUDFLARE_API_TOKEN')

if not API_TOKEN:
    import getpass
    API_TOKEN = getpass.getpass('Enter your Cloudflare API token: ').strip()

assert API_TOKEN, 'A Cloudflare API token is required.'
print('Token loaded (length: %d).' % len(API_TOKEN))

## 3. Configuration & API client

`LOCATION='IR'` targets Iran. `DATE_RANGE` controls the look-back window
(e.g. `'7d'`, `'28d'`, `'52w'`). Widen it to capture a known unrest period.

In [ ]:
BASE_URL    = 'https://api.cloudflare.com/client/v4/radar'
LOCATION    = 'IR'      # ISO alpha-2 code for Iran
DATE_RANGE  = '28d'     # look-back window
AGG_INTERVAL = '1h'     # 1h gives enough resolution to see shutdowns

HEADERS = {'Authorization': f'Bearer {API_TOKEN}', 'Content-Type': 'application/json'}


def radar_get(path: str, params: Optional[dict] = None) -> dict:
    """GET a Cloudflare Radar endpoint and return the `result` payload."""
    resp = requests.get(f'{BASE_URL}/{path.lstrip("/")}', headers=HEADERS,
                        params=params or {}, timeout=30)
    if resp.status_code != 200:
        raise RuntimeError(f'{resp.status_code} for {path}: {resp.text[:400]}')
    payload = resp.json()
    if not payload.get('success', False):
        raise RuntimeError(f'API error for {path}: {payload.get("errors")}')
    return payload['result']


def series_to_frame(result: dict, value_name: str = 'value') -> pd.DataFrame:
    """Convert a Radar timeseries `result` (serie_0) into a tidy DataFrame."""
    serie = result.get('serie_0') or next(
        (v for k, v in result.items() if k.startswith('serie')), None)
    if not serie:
        raise ValueError('No timeseries found in result.')
    df = pd.DataFrame({
        'timestamp': pd.to_datetime(serie['timestamps'], utc=True),
        value_name: pd.to_numeric(serie['values'], errors='coerce'),
    })
    return df.set_index('timestamp').sort_index()

print('Client ready. Target:', LOCATION, '| range:', DATE_RANGE)

## 4. HTTP request traffic

Relative HTTP request volume from Iran. Healthy traffic shows a smooth diurnal
(day/night) cycle; a near-vertical cliff that doesn't match that rhythm is the
classic signature of a shutdown.

In [ ]:
http_res = radar_get('http/timeseries', {
    'location': LOCATION,
    'dateRange': DATE_RANGE,
    'aggInterval': AGG_INTERVAL,
})
http_df = series_to_frame(http_res, 'http_requests')
print(f'{len(http_df)} points from {http_df.index.min()} to {http_df.index.max()}')
http_df.tail()

In [ ]:
ax = http_df['http_requests'].plot(color='#1f77b4', lw=1.2)
ax.set_title(f'Iran — HTTP request volume (relative), last {DATE_RANGE}')
ax.set_ylabel('Normalized requests')
ax.set_xlabel('')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout(); plt.show()

## 5. Network traffic (NetFlows)

A second, independent volume signal derived from network flow data. Cross-checking
HTTP against NetFlows reduces the chance that a dip is just a measurement artifact.

In [ ]:
try:
    nf_res = radar_get('netflows/timeseries', {
        'location': LOCATION,
        'dateRange': DATE_RANGE,
        'aggInterval': AGG_INTERVAL,
        'product': 'HTTP',
    })
    nf_df = series_to_frame(nf_res, 'netflows')
    ax = nf_df['netflows'].plot(color='#2ca02c', lw=1.2)
    ax.set_title(f'Iran — NetFlows traffic (relative), last {DATE_RANGE}')
    ax.set_ylabel('Normalized traffic'); ax.set_xlabel('')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.tight_layout(); plt.show()
except RuntimeError as e:
    nf_df = None
    print('NetFlows unavailable for this token/range:', e)

## 6. Anomaly detection — flag abnormal traffic drops

We compare each point to a rolling **median baseline** (robust to spikes) and the
rolling **MAD** (median absolute deviation). A point is flagged as a *suspected
disruption* when traffic falls well below baseline by both a relative threshold
(e.g. <50% of baseline) and a statistical one (a strong negative robust z-score).

Tune `DROP_RATIO` and `Z_THRESHOLD` for sensitivity.

In [ ]:
WINDOW      = 24      # baseline window in points (24 = ~1 day at 1h interval)
DROP_RATIO  = 0.50    # flag if value < 50% of rolling-median baseline
Z_THRESHOLD = 3.5     # robust z-score magnitude for a 'large' deviation


def detect_drops(df: pd.DataFrame, col: str) -> pd.DataFrame:
    s = df[col].astype(float)
    baseline = s.rolling(WINDOW, min_periods=WINDOW // 2, center=True).median()
    mad = (s - baseline).abs().rolling(WINDOW, min_periods=WINDOW // 2,
                                       center=True).median()
    robust_z = 0.6745 * (s - baseline) / mad.replace(0, pd.NA)
    out = pd.DataFrame({
        col: s, 'baseline': baseline, 'robust_z': robust_z,
        'pct_of_baseline': s / baseline,
    })
    out['disruption'] = (out['pct_of_baseline'] < DROP_RATIO) & (out['robust_z'] < -Z_THRESHOLD)
    return out


flagged = detect_drops(http_df, 'http_requests')
n_flags = int(flagged['disruption'].sum())
print(f'Suspected disruption points: {n_flags}')
flagged[flagged['disruption']].head(20)

In [ ]:
ax = flagged['http_requests'].plot(color='#1f77b4', lw=1.1, label='HTTP requests')
flagged['baseline'].plot(ax=ax, color='#888', lw=1.0, ls='--', label='rolling baseline')
hits = flagged[flagged['disruption']]
ax.scatter(hits.index, hits['http_requests'], color='red', s=28, zorder=5,
           label='suspected disruption')
ax.set_title('Iran — HTTP traffic with flagged drops')
ax.set_ylabel('Normalized requests'); ax.set_xlabel('')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.legend(loc='lower left'); plt.tight_layout(); plt.show()

### Group flagged points into disruption windows

Consecutive flagged hours are merged into a single event with start, end and a
severity estimate (how far traffic fell below baseline).

In [ ]:
def group_windows(flagged_df: pd.DataFrame, gap: str = '2h') -> pd.DataFrame:
    hits = flagged_df[flagged_df['disruption']].copy()
    if hits.empty:
        return pd.DataFrame(columns=['start', 'end', 'duration_h', 'min_pct_of_baseline'])
    gap_td = pd.Timedelta(gap)
    new_event = hits.index.to_series().diff() > gap_td
    hits['event'] = new_event.cumsum()
    rows = []
    for _, grp in hits.groupby('event'):
        rows.append({
            'start': grp.index.min(),
            'end': grp.index.max(),
            'duration_h': round((grp.index.max() - grp.index.min()).total_seconds() / 3600 + 1, 1),
            'min_pct_of_baseline': round(float(grp['pct_of_baseline'].min()) * 100, 1),
        })
    return pd.DataFrame(rows)


windows = group_windows(flagged)
windows

## 7. Cloudflare outage annotations

Cloudflare curates a list of confirmed internet outages with cause and scope.
Cross-referencing your flagged windows against these annotations is the strongest
single corroboration available inside Radar.

In [ ]:
out_res = radar_get('annotations/outages', {
    'location': LOCATION,
    'dateRange': '52w',   # outages are rarer; use a wide window
    'limit': 50,
})
annotations = out_res.get('annotations', [])
print(f'{len(annotations)} outage annotation(s) for {LOCATION}.')

if annotations:
    out_df = pd.json_normalize(annotations)
    keep = [c for c in ['startDate', 'endDate', 'eventType', 'outage.outageCause',
                        'outage.outageType', 'description', 'linkedUrl']
            if c in out_df.columns]
    display(out_df[keep])
else:
    print('No curated outages in this window (absence is not proof of none).')

## 8. Internet Quality Index (secondary signal)

During throttling (as opposed to a full cutoff), volume may stay up while
**latency rises** and **bandwidth falls**. The IQI captures that.

In [ ]:
try:
    iqi_res = radar_get('quality/iqi/timeseries_groups', {
        'location': LOCATION,
        'dateRange': DATE_RANGE,
        'metric': 'latency',
    })
    iqi_df = series_to_frame(iqi_res, 'latency_ms')
    ax = iqi_df['latency_ms'].plot(color='#d62728', lw=1.2)
    ax.set_title(f'Iran — Internet quality: latency, last {DATE_RANGE}')
    ax.set_ylabel('Latency (ms, p25/median est.)'); ax.set_xlabel('')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    plt.tight_layout(); plt.show()
except RuntimeError as e:
    print('IQI unavailable for this token/range:', e)

## 9. Summary

Pull the findings together into a short report.

In [ ]:
print('=' * 64)
print(f'  Iran connectivity report — generated {dt.datetime.utcnow():%Y-%m-%d %H:%M} UTC')
print('=' * 64)
print(f'Window analysed     : {http_df.index.min():%Y-%m-%d} -> {http_df.index.max():%Y-%m-%d}')
print(f'Data points (HTTP)  : {len(http_df)}')
print(f'Flagged drop hours  : {int(flagged["disruption"].sum())}')
print(f'Disruption windows  : {len(windows)}')
print(f'Curated outages     : {len(annotations)}')
print('-' * 64)
if len(windows):
    print('Suspected disruption windows:')
    for _, w in windows.iterrows():
        print(f'  - {w.start:%Y-%m-%d %H:%M} -> {w.end:%H:%M} UTC '
              f'({w.duration_h}h, low {w.min_pct_of_baseline}% of baseline)')
else:
    print('No abnormal traffic drops flagged in this window.')
print('-' * 64)
print('Reminder: corroborate against NetBlocks / OONI / news before concluding intent.')

---
### Next steps & references
- Widen `DATE_RANGE` to cover a specific unrest period and re-run.
- Break down by **ASN** (mobile vs fixed-line operators) to spot targeted shutdowns.
- Corroborate with: [Cloudflare Radar — Iran](https://radar.cloudflare.com/ir),
  [NetBlocks](https://netblocks.org/), [OONI Explorer](https://explorer.ooni.org/),
  [IODA](https://ioda.inetintel.cc.gatech.edu/).
- API docs: <https://developers.cloudflare.com/radar/>